# NLP Lab 5: Text Representation

**Student ID:** 2240004223  
**Topics:** Cosine Similarity, TF-IDF, and Skip-Gram Word2Vec

## Setup

The following cell installs missing packages when necessary and imports all libraries used in the lab.

In [1]:
import importlib.util
import subprocess
import sys
required_packages = ["gensim", "nltk"]
missing_packages = [package for package in required_packages if importlib.util.find_spec(package) is None]
if missing_packages:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing_packages], check=True)

In [2]:
import re
from pathlib import Path
import gensim
import numpy as np
import pandas as pd
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## Task 1: Cosine Similarity

Determine how similar the four supplied sentences are.

In [3]:
documents = [
    "This is the first document.",
    "This document is the second document.",
    "And this is the third one.",
    "Is this the first document?"
]
document_labels = [f"Sentence {number}" for number in range(1, len(documents) + 1)]
similarity_vectorizer = TfidfVectorizer()
document_vectors = similarity_vectorizer.fit_transform(documents)
similarity_values = cosine_similarity(document_vectors)
similarity_matrix = pd.DataFrame(
    similarity_values,
    index=document_labels,
    columns=document_labels
).round(4)
similarity_matrix.index.name = "Sentence"
similarity_matrix

,Sentence 1,Sentence 2,Sentence 3,Sentence 4
Sentence,,,,
Sentence 1,1.0000,0.6469,0.3078,1.0000
Sentence 2,0.6469,1.0000,0.2252,0.6469
Sentence 3,0.3078,0.2252,1.0000,0.3078
Sentence 4,1.0000,0.6469,0.3078,1.0000


In [4]:
similarity_rows = []
for left_index in range(len(document_labels)):
    for right_index in range(left_index + 1, len(document_labels)):
        similarity_rows.append({
            "Sentence Pair": f"{document_labels[left_index]} and {document_labels[right_index]}",
            "Cosine Similarity": similarity_values[left_index, right_index]
        })
similarity_ranking = pd.DataFrame(similarity_rows).sort_values(
    "Cosine Similarity", ascending=False
).reset_index(drop=True)
similarity_ranking["Cosine Similarity"] = similarity_ranking["Cosine Similarity"].round(4)
similarity_ranking

,Sentence Pair,Cosine Similarity
0,Sentence 1 and Sentence 4,1.0000
1,Sentence 1 and Sentence 2,0.6469
2,Sentence 2 and Sentence 4,0.6469
3,Sentence 1 and Sentence 3,0.3078
4,Sentence 3 and Sentence 4,0.3078
5,Sentence 2 and Sentence 3,0.2252


### Task 1 Result

Sentence 1 and Sentence 4 are the most similar distinct pair. Their cosine similarity is **1.0000** because they contain the same terms after normalization.

## Task 2: TF-IDF

Apply TF-IDF and determine the most important words in each supplied sentence.

In [5]:
tfidf_documents = [
    "data science is one of the most important fields of science",
    "this is one of the best data science courses",
    "data scientists analyze data"
]
tfidf_labels = [f"Sentence {number}" for number in range(1, len(tfidf_documents) + 1)]
tfidf_vectorizer = TfidfVectorizer()
tfidf_values = tfidf_vectorizer.fit_transform(tfidf_documents)
tfidf_matrix = pd.DataFrame(
    tfidf_values.toarray(),
    index=tfidf_labels,
    columns=tfidf_vectorizer.get_feature_names_out()
).round(4)
tfidf_matrix.index.name = "Sentence"
tfidf_matrix

,analyze,best,courses,data,fields,important,is,most,of,one,science,scientists,the,this
Sentence,,,,,,,,,,,,,,
Sentence 1,0.0000,0.0000,0.0000,0.1895,0.3209,0.3209,0.2440,0.3209,0.4881,0.2440,0.4881,0.0000,0.2440,0.0000
Sentence 2,0.0000,0.4003,0.4003,0.2364,0.0000,0.0000,0.3044,0.0000,0.3044,0.3044,0.3044,0.0000,0.3044,0.4003
Sentence 3,0.5427,0.0000,0.0000,0.6411,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.5427,0.0000,0.0000


In [6]:
important_word_rows = []
for sentence_index, sentence_label in enumerate(tfidf_labels):
    scores = pd.Series(
        tfidf_values.toarray()[sentence_index],
        index=tfidf_vectorizer.get_feature_names_out()
    )
    for rank, (word, score) in enumerate(scores[scores > 0].sort_values(ascending=False).head(5).items(), start=1):
        important_word_rows.append({
            "Sentence": sentence_label,
            "Rank": rank,
            "Word": word,
            "TF-IDF": round(score, 4)
        })
important_words = pd.DataFrame(important_word_rows).set_index(["Sentence", "Rank"])
important_words

Word  TF-IDF
Sentence   Rank                    
Sentence 1 1        science  0.4881
           2             of  0.4881
           3           most  0.3209
           4         fields  0.3209
           5      important  0.3209
Sentence 2 1           best  0.4003
           2        courses  0.4003
           3           this  0.4003
           4             of  0.3044
           5             is  0.3044
Sentence 3 1           data  0.6411
           2        analyze  0.5427
           3     scientists  0.5427

### Task 2 Result

The ranking table shows the five highest non-zero TF-IDF terms for each sentence. Sentence-specific words receive larger weights, while terms appearing across multiple sentences receive smaller weights.

## Task 3: Simpsons Preprocessing and Skip-Gram Word2Vec

Load `simpsons_script_lines.csv`, clean the `spoken_words` column with the required preprocessing function, tokenize the text, and train the requested skip-gram model.

In [7]:
data_paths = [
    Path("simpsons_script_lines.csv"),
    Path("upload/simpsons_script_lines.csv"),
    Path("/content/simpsons_script_lines.csv")
]
data_path = next(path for path in data_paths if path.exists())
simpsons = pd.read_csv(data_path)
simpsons[["raw_character_text", "spoken_words"]].head()

,raw_character_text,spoken_words
0,Miss Hoover,"No, actually, it was a little of both. Sometim..."
1,Lisa Simpson,Where's Mr. Bergstrom?
2,Miss Hoover,I don't know. Although I'd sure like to talk t...
3,Lisa Simpson,That life is worth living.
4,Edna Krabappel-Flanders,The polls will be open from now until the end ...


In [8]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[0-9]", "", text)
    text = re.sub(r"[)(,”“.’$-]", "", text)
    return text
cleaned_text = simpsons["spoken_words"].dropna().astype(str).map(clean_text)
tokens = [
    [token for token in word_tokenize(text, preserve_line=True) if token.isalpha()]
    for text in cleaned_text
]
tokens = [sentence for sentence in tokens if sentence]
preprocessing_summary = pd.DataFrame({
    "Measure": ["Original rows", "Non-null spoken lines", "Tokenized lines", "Total tokens"],
    "Value": [len(simpsons), simpsons["spoken_words"].notna().sum(), len(tokens), sum(len(sentence) for sentence in tokens)]
})
preprocessing_summary

,Measure,Value
0,Original rows,158314
1,Non-null spoken lines,131855
2,Tokenized lines,131699
3,Total tokens,1299321


In [9]:
cleaning_examples = pd.DataFrame({
    "Original": simpsons.loc[simpsons["spoken_words"].notna(), "spoken_words"].head(5).tolist(),
    "Cleaned": cleaned_text.head(5).tolist(),
    "Tokens": tokens[:5]
})
cleaning_examples

,Original,Cleaned,Tokens
0,"No, actually, it was a little of both. Sometim...",no actually it was a little of both sometimes ...,"[no, actually, it, was, a, little, of, both, s..."
1,Where's Mr. Bergstrom?,where's mr bergstrom?,"[where, mr, bergstrom]"
2,I don't know. Although I'd sure like to talk t...,i don't know although i'd sure like to talk to...,"[i, do, know, although, i, sure, like, to, tal..."
3,That life is worth living.,that life is worth living,"[that, life, is, worth, living]"
4,The polls will be open from now until the end ...,the polls will be open from now until the end ...,"[the, polls, will, be, open, from, now, until,..."


In [10]:
Skip_gram_model = gensim.models.Word2Vec(
    tokens,
    min_count=1,
    vector_size=100,
    window=5,
    sg=1,
    workers=1,
    seed=42
)
model_summary = pd.DataFrame({
    "Setting": ["Architecture", "Vocabulary size", "Vector size", "Window", "Minimum count"],
    "Value": ["Skip-Gram", len(Skip_gram_model.wv), Skip_gram_model.vector_size, Skip_gram_model.window, Skip_gram_model.min_count]
})
model_summary

,Setting,Value
0,Architecture,Skip-Gram
1,Vocabulary size,43451
2,Vector size,100
3,Window,5
4,Minimum count,1


### Task 3 Result

The text was lowercased, digits and the specified punctuation were removed, empty values were excluded, and the remaining lines were tokenized. The skip-gram Word2Vec model was trained with 100-dimensional vectors, a context window of 5, and a minimum count of 1.

## Task 4: Similar Words

Use `wv.most_similar()` to find words similar to **homer**, **marge**, and **bart**.

In [11]:
similar_word_rows = []
for target_word in ["homer", "marge", "bart"]:
    for rank, (similar_word, similarity) in enumerate(Skip_gram_model.wv.most_similar(target_word, topn=10), start=1):
        similar_word_rows.append({
            "Target Word": target_word,
            "Rank": rank,
            "Similar Word": similar_word,
            "Similarity": round(similarity, 4)
        })
similar_words = pd.DataFrame(similar_word_rows).set_index(["Target Word", "Rank"])
similar_words

Similar Word  Similarity
Target Word Rank                         
homer       1             abe      0.8674
            2           marge      0.8495
            3     bartholomew      0.8277
            4          grampa      0.8250
            5             jay      0.8240
            6           eliza      0.8178
            7          mister      0.8103
            8         jessica      0.8060
            9      sweetheart      0.8041
            10          monty      0.8023
marge       1           homer      0.8495
            2             abe      0.8402
            3         sweetie      0.8188
            4           becky      0.8130
            5           honey      0.8121
            6            lisa      0.8040
            7           midge      0.8038
            8      sweetheart      0.8037
            9            edna      0.7998
            10      carefully      0.7960
bart        1            lisa      0.8538
            2        milhouse      0.8301
            3         jessica      0.8243
            4          grampa      0.8224
            5           eliza      0.8120
            6             abe      0.8085
            7          laddie      0.8070
            8          nelson      0.8058
            9         allison      0.8035
            10          homer      0.8009

## Task 5: Odd Word

Use `wv.doesnt_match()` to find the word that does not belong in each group.

In [12]:
word_groups = [
    ["jimbo", "milhouse", "kearney"],
    ["nelson", "bart", "milhouse"],
    ["homer", "patty", "selma"]
]
odd_words = pd.DataFrame({
    "Word Group": [", ".join(group) for group in word_groups],
    "Odd Word": [Skip_gram_model.wv.doesnt_match(group) for group in word_groups]
})
odd_words

,Word Group,Odd Word
0,"jimbo, milhouse, kearney",milhouse
1,"nelson, bart, milhouse",nelson
2,"homer, patty, selma",homer


## Conclusion

TF-IDF represented documents according to term importance, cosine similarity compared document vectors, and skip-gram Word2Vec learned word relationships from the Simpsons dialogue. The trained model was then used for similarity and odd-word analysis.